# DeFoG — Colab Environment Setup (run once)

This notebook builds the `defog` conda environment **directly on the Colab GPU instance**
and saves it to Google Drive as a compressed tarball for fast restore in future sessions.

**Run this notebook once (or whenever dependencies change).**  
For every subsequent session, open and run `02_session_init.ipynb` instead.

### Before running
1. Set the runtime to **GPU** (Runtime → Change runtime type → T4/A100).
2. Set `GITHUB_REPO` in cell 4 to your fork's URL.
3. Upload the two precomputed `.pt` files to Drive **after** cell 2 completes:
   - `MyDrive/DeFoGColab/data/qm9/qm9_motif_marginals_6x1.pt`
   - `MyDrive/DeFoGColab/data/qm9/spminer_motif_marginals.pt`

### Expected total runtime
~25–35 min (graph-tool conda solve: ~10 min, conda-pack: ~8 min, Drive upload: ~5 min).

In [ ]:
# ── Cell 1: Mount Drive and create folder structure ───────────────────────────
from google.colab import drive
import os

drive.mount('/content/drive')

DRIVE_BASE = '/content/drive/MyDrive/DeFoGColab'
for d in [
    f'{DRIVE_BASE}',
    f'{DRIVE_BASE}/data/qm9',
    f'{DRIVE_BASE}/checkpoints',
]:
    os.makedirs(d, exist_ok=True)

print('Drive mounted.')
print(f'DeFoGColab root: {DRIVE_BASE}')
print('Folder structure created.')

In [ ]:
# ── Cell 2: Verify GPU and CUDA ───────────────────────────────────────────────
# The CUDA version shown here determines which PyTorch wheel will be installed.
# colab_create_env.sh detects this automatically.
!nvidia-smi

In [ ]:
%%bash
# ── Cell 3: Install Miniconda ─────────────────────────────────────────────────
set -e
if [ -f /content/miniconda3/bin/conda ]; then
    echo "Miniconda already present, skipping install."
    /content/miniconda3/bin/conda --version
    exit 0
fi
echo "Downloading Miniconda..."
wget -q https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh \
     -O /tmp/miniconda.sh
bash /tmp/miniconda.sh -b -p /content/miniconda3
rm /tmp/miniconda.sh
echo "Miniconda installed."
/content/miniconda3/bin/conda --version

In [ ]:
# ── Cell 4: Configure — set your GitHub repo URL here ─────────────────────────
GITHUB_REPO = 'https://github.com/<YOUR_USERNAME>/DeFoGPlus.git'
REPO_DIR    = '/content/DeFoGPlus'

print(f'Will clone: {GITHUB_REPO}')
print(f'       to : {REPO_DIR}')

In [ ]:
%%bash -s "$GITHUB_REPO" "$REPO_DIR"
# ── Cell 5: Clone repo ────────────────────────────────────────────────────────
GITHUB_REPO="$1"
REPO_DIR="$2"
set -e
if [ -d "$REPO_DIR/.git" ]; then
    echo "Repo already present, pulling latest."
    git -C "$REPO_DIR" pull
else
    git clone "$GITHUB_REPO" "$REPO_DIR"
fi
echo "Repo ready at $REPO_DIR"
ls "$REPO_DIR"

In [ ]:
%%bash
# ── Cell 6: Create defog conda environment ────────────────────────────────────
# This is the longest step (~15-20 min).
# graph-tool from conda-forge takes ~10 min to solve and download.
set -e
bash /content/DeFoGPlus/shell/colab_create_env.sh 2>&1

In [ ]:
# ── Cell 7: Smoke test — verify key imports ───────────────────────────────────
import subprocess, os

PYTHON = '/content/miniconda3/envs/defog/bin/python'

check = """
import graph_tool
import torch
import rdkit
import torch_geometric
import pytorch_lightning
import hydra
import wandb
print('graph_tool       :', graph_tool.__version__)
print('torch            :', torch.__version__)
print('CUDA available   :', torch.cuda.is_available())
print('torch_geometric  :', torch_geometric.__version__)
print('pytorch_lightning:', pytorch_lightning.__version__)
print('All imports OK.')
"""

# Set DISPLAY so graph_tool does not attempt to open an X window.
env = {**os.environ, 'MPLBACKEND': 'Agg'}
r = subprocess.run([PYTHON, '-c', check], capture_output=True, text=True, env=env)
print(r.stdout)
if r.returncode != 0:
    print('STDERR:', r.stderr)
    raise RuntimeError('Smoke test failed — check the output above.')

In [ ]:
%%bash
# ── Cell 8: Pack environment to Drive ────────────────────────────────────────
# conda-pack creates a self-contained tarball of the entire environment,
# including both conda and pip packages. conda-unpack (run at restore time)
# rewrites any baked-in absolute paths.
#
# Expected tarball size: 2-4 GB compressed.
# Expected runtime: 5-10 min (packing) + 3-8 min (Drive upload).
set -e
TARBALL_TMP='/tmp/defog_env.tar.gz'
TARBALL_DST='/content/drive/MyDrive/DeFoGColab/defog_env.tar.gz'

echo "Packing environment — this will take several minutes..."
/content/miniconda3/envs/defog/bin/conda-pack \
    -n defog \
    -o "$TARBALL_TMP" \
    --ignore-missing-files

echo "Pack complete. Size: $(du -sh $TARBALL_TMP | cut -f1)"
echo "Copying to Drive..."
cp "$TARBALL_TMP" "$TARBALL_DST"
rm "$TARBALL_TMP"
echo "Uploaded: $(du -sh $TARBALL_DST | cut -f1)  →  $TARBALL_DST"

## ✅ Setup complete

The environment tarball is now saved at:
```
MyDrive/DeFoGColab/defog_env.tar.gz
```

### Next: upload the precomputed `.pt` files to Drive

In the Drive web UI, upload these two files to `MyDrive/DeFoGColab/data/qm9/`:
- `qm9_motif_marginals_6x1.pt`  (ring-edited marginals)
- `spminer_motif_marginals.pt`  (SPMiner-mined marginals)

They will be accessible through the `data/` symlink created in every session by `02_session_init.ipynb`.

From now on, start each new Colab session with **`02_session_init.ipynb`**.